In [0]:
%sql
create catalog if not exists realtime_weather;
use catalog realtime_weather;

In [0]:
%sql
create schema if not exists bronze;
create schema if not exists silver;
create schema if not exists gold;
create schema if not exists config;
CREATE SCHEMA IF NOT EXISTS monitoring;


In [0]:
import requests
import json
from datetime import datetime
from pyspark.sql import SparkSession

# spark = SparkSession.builder.getOrCreate()

# Load config
config = json.load(open('/Volumes/realtime_weather/default/config/config.json'))

api_key = config['api_key']
cities = config['cities']
base_url = config['api_base_url']

data = []
#use multithreading method
for city in cities:
    url = f"{base_url}?q={city}&appid={api_key}&units=metric"
    
    start = datetime.now()
    response = requests.get(url)
    latency = (datetime.now() - start).total_seconds() * 1000

    if response.status_code == 200:
        raw_json = response.text
    else:
        raw_json = None

    data.append((
        city,
        response.status_code,
        raw_json,
        datetime.now(),
        latency
    ))

# Create DataFrame
api_df = spark.createDataFrame(data, [
    "city_requested",
    "http_status",
    "raw_response",
    "call_timestamp",
    "response_latency_ms"
])

# Add load_date
from pyspark.sql.functions import current_date
api_df = api_df.withColumn("load_date", current_date())

# Write to Delta
api_df.write.format("delta") \
    .mode("append") \
    .partitionBy("load_date") \
    .saveAsTable("realtime_weather.bronze.bronze_weather_data")

In [0]:
# %sql
# drop table if exists realtime_weather.bronze.bronze_weather_data

In [0]:
df1 = spark.read.table("realtime_weather.bronze.bronze_weather_data")
display(df1)

city_requested,http_status,raw_response,call_timestamp,response_latency_ms,load_date
Chennai,200,"{""coord"":{""lon"":80.2785,""lat"":13.0878},""weather"":[{""id"":801,""main"":""Clouds"",""description"":""few clouds"",""icon"":""02d""}],""base"":""stations"",""main"":{""temp"":35.1,""feels_like"":42.1,""temp_min"":34.98,""temp_max"":36.24,""pressure"":1005,""humidity"":60,""sea_level"":1005,""grnd_level"":1005},""visibility"":6000,""wind"":{""speed"":7.72,""deg"":170},""clouds"":{""all"":20},""dt"":1777454829,""sys"":{""type"":2,""id"":2104103,""country"":""IN"",""sunrise"":1777421939,""sunset"":1777467201},""timezone"":19800,""id"":1264527,""name"":""Chennai"",""cod"":200}",2026-04-29T09:30:34.757Z,70.39,2026-04-29
Mumbai,200,"{""coord"":{""lon"":72.8479,""lat"":19.0144},""weather"":[{""id"":721,""main"":""Haze"",""description"":""haze"",""icon"":""50d""}],""base"":""stations"",""main"":{""temp"":32.99,""feels_like"":38.79,""temp_min"":31.94,""temp_max"":32.99,""pressure"":1006,""humidity"":58,""sea_level"":1006,""grnd_level"":1006},""visibility"":6000,""wind"":{""speed"":7.72,""deg"":230},""clouds"":{""all"":0},""dt"":1777454683,""sys"":{""type"":1,""id"":9052,""country"":""IN"",""sunrise"":1777423318,""sunset"":1777469388},""timezone"":19800,""id"":1275339,""name"":""Mumbai"",""cod"":200}",2026-04-29T09:30:34.831Z,74.426,2026-04-29
Delhi,200,"{""coord"":{""lon"":77.2167,""lat"":28.6667},""weather"":[{""id"":802,""main"":""Clouds"",""description"":""scattered clouds"",""icon"":""03d""}],""base"":""stations"",""main"":{""temp"":36.05,""feels_like"":35.76,""temp_min"":36.05,""temp_max"":36.05,""pressure"":1000,""humidity"":28,""sea_level"":1000,""grnd_level"":976},""visibility"":6000,""wind"":{""speed"":3.09,""deg"":80},""clouds"":{""all"":40},""dt"":1777454982,""sys"":{""type"":1,""id"":9165,""country"":""IN"",""sunrise"":1777421535,""sunset"":1777469075},""timezone"":19800,""id"":1273294,""name"":""Delhi"",""cod"":200}",2026-04-29T09:30:34.902Z,71.039,2026-04-29
Bangalore,200,"{""coord"":{""lon"":77.6033,""lat"":12.9762},""weather"":[{""id"":802,""main"":""Clouds"",""description"":""scattered clouds"",""icon"":""03d""}],""base"":""stations"",""main"":{""temp"":34.72,""feels_like"":36.7,""temp_min"":32.9,""temp_max"":35.76,""pressure"":1005,""humidity"":40,""sea_level"":1005,""grnd_level"":910},""visibility"":6000,""wind"":{""speed"":6.71,""deg"":108,""gust"":22.8},""clouds"":{""all"":40},""dt"":1777454643,""sys"":{""type"":2,""id"":2017753,""country"":""IN"",""sunrise"":1777422588,""sunset"":1777467836},""timezone"":19800,""id"":1277333,""name"":""Bengaluru"",""cod"":200}",2026-04-29T09:30:34.970Z,67.258,2026-04-29
Hyderabad,200,"{""coord"":{""lon"":78.4744,""lat"":17.3753},""weather"":[{""id"":801,""main"":""Clouds"",""description"":""few clouds"",""icon"":""02d""}],""base"":""stations"",""main"":{""temp"":38.23,""feels_like"":38.85,""temp_min"":38.23,""temp_max"":38.73,""pressure"":1003,""humidity"":27,""sea_level"":1003,""grnd_level"":942},""visibility"":6000,""wind"":{""speed"":4.12,""deg"":260},""clouds"":{""all"":20},""dt"":1777454707,""sys"":{""type"":1,""id"":9214,""country"":""IN"",""sunrise"":1777422083,""sunset"":1777467923},""timezone"":19800,""id"":1269843,""name"":""Hyderabad"",""cod"":200}",2026-04-29T09:30:35.042Z,72.635,2026-04-29
Kolkata,200,"{""coord"":{""lon"":88.3697,""lat"":22.5697},""weather"":[{""id"":721,""main"":""Haze"",""description"":""haze"",""icon"":""50d""}],""base"":""stations"",""main"":{""temp"":28.97,""feels_like"":32.66,""temp_min"":28.97,""temp_max"":28.97,""pressure"":1002,""humidity"":70,""sea_level"":1002,""grnd_level"":1002},""visibility"":4000,""wind"":{""speed"":7.72,""deg"":100},""clouds"":{""all"":75},""dt"":1777454950,""sys"":{""type"":1,""id"":9114,""country"":""IN"",""sunrise"":1777419338,""sunset"":1777465919},""timezone"":19800,""id"":1275004,""name"":""Kolkata"",""cod"":200}",2026-04-29T09:30:35.110Z,67.476,2026-04-29
Pune,200,"{""coord"":{""lon"":73.8553,""lat"":18.5196},